## Team overview

Fill in the table, one row per team member. The question should be the same one from your proposal, or the updated version if it changed.

| Team member | Question they will answer | Changed since the proposal? (Yes / No) |
|---|---|---|
| Yujin Tatar |Is there a correlation between Social Engagement Index and where Divvy trips and stations are concentrated?  |No |
| Anah Shaikh |Does the most prominent type of bike lane (protected, buffered, etc) in a community affect bike usage and which types of lanes have the most crashes? How does this data compare against the hardship index of each area? |No |
| Wiktoria Mleczko |Does the share of trips by bike type (electric, classic) differ between Divvy stations inside Equity Priority Areas and those outside them? Is there still a difference when adjusting for the number of stations?|No| 
| Luke Hallum |Which major Chicago special events change bike-share demand the most?  How much does each event change ridership versus a normal day, and which stations and which stations need to have additional bikes? |No |

**Presentation session:** Session 1 (Wed, Nov 25) 

**If any question changed since the proposal, explain why in one or two sentences:**

In [ ]:
# Libraries used in this notebook. Add any others your team needs.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely import wkt
from haversine import Unit, haversine

## 1) Data loaded and cleaned
*By \<Wiktoria Mleczko, Anah Shaikh, Yujin Tatar, Luke Hallum>*

Load your data and show that it is ready (or nearly ready) to analyze.

If you use more than one dataset, load each one and show how they are combined.

In [ ]:
# Load the data. Use a relative path (for example, a "data" folder next to this notebook)
# so the notebook runs on any teammate's computer.

#Q1 (Yujin):
trip_path = "data/202609-divvy-tripdata.csv"
stations_path = "data/Divvy_Bicycle_Stations_20261117.csv"
sei_path = "data/Chicago Health Atlas.csv"
trip_data = pd.read_csv(trip_path)
stations = pd.read_csv(stations_path)
sei = pd.read_csv(sei_path)

# Size and first rows
print(trip_data.shape)
display(trip_data.head())
print(stations.shape)
display(stations.head())
print(sei.shape)
display(sei.head())

# Column types and missing values
trip_data.info()
trip_data.isna().sum().sort_values(ascending=False)
stations.info()
stations.isna().sum().sort_values(ascending=False)
sei.info()
sei.isna().sum().sort_values(ascending=False)

# Q2 (ANAH)

# bike routes (I will be looking at location and types of bike lanes)
bike_routes = pd.read_csv('data/Bike_Routes_20261123.csv')

# crashes
bike_crashes = pd.read_csv('data/Traffic_Crashes_-_Crashes_20261123.csv')

# community boundaries
communities = pd.read_csv('data/Boundaries_-_Community_Areas_20261123.csv')

# socioeconomic indicators
census = pd.read_csv('data/Census_Data_-_Selected_socioeconomic_indicators_in_Chicago,_2008_–_2012_20261123.csv')

# Q3 (Wiktoria)

# Reading the divvy dataset
divvy_trips_= "data/202609-divvy-tripdata.csv"
divvy_trips_df = pd.read_csv(divvy_trips)

# Showing the shape of the dataset
print(divvy_trips_df.shape)
divvy_trips_df.head()

# Reading the equity priority area (EPA) dataset 
epa = "data/Divvy_Equity_Priority_Area_20261121.csv"
epa_df = gpd.read_file(epa)

# Showing the shape of the dataset
print(epa_df.shape)
epa_df.head()

# Identifying the missing values 
divvy_trips_df.isna().sum()
# Identifying duplicate rows 
print(divvy_trips_df.duplicated().sum())

#Checking missing values in EPA dataset
print(epa.isna().sum())
print(epa.geometry.isna().sum())

# Q4 (Luke)
# read each month of divvy data going back one calendar year
Nov = pd.read_csv('data/Divvy Data/202511-divvy-tripdata.csv')
Dec = pd.read_csv('data/Divvy Data/202512-divvy-tripdata.csv')
Jan = pd.read_csv('data/Divvy Data/202601-divvy-tripdata.csv')
Feb = pd.read_csv('data/Divvy Data/202602-divvy-tripdata.csv')
Mar = pd.read_csv('data/Divvy Data/202603-divvy-tripdata.csv')
Apr = pd.read_csv('data/Divvy Data/202604-divvy-tripdata.csv')
May = pd.read_csv('data/Divvy Data/202605-divvy-tripdata.csv')
Jun = pd.read_csv('data/Divvy Data/202606-divvy-tripdata.csv')
Jul = pd.read_csv('data/Divvy Data/202607-divvy-tripdata.csv')
Aug = pd.read_csv('data/Divvy Data/202608-divvy-tripdata.csv')
Sep = pd.read_csv('data/Divvy Data/202609-divvy-tripdata.csv')
Oct = pd.read_csv('data/Divvy Data/202610-divvy-tripdata.csv')

rides = pd.concat(
    [Nov, Dec, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct], 
ignore_index = True) 

# Read the Bears data sets and combine them
B1 = pd.read_csv('data/bears1.csv')
B2 = pd.read_csv('data/bears2.csv')

NFL = pd.concat([B1, B2], ignore_index=True)

# Filter to Home games
Home = (NFL['Location'] == 'Soldier Field') 
Bears = NFL[Home].copy()

NBA = pd.read_csv('data/Bulls.csv')
Home2 = (NBA['Location'] =='United Center')
Bulls = NBA[Home2].copy()

Cubs = pd.read_csv('data/Cubs.csv')

In [ ]:
# Cleaning steps done so far. Add a comment above each step that says what it does and why.

# Q1 (Yujin)
# Drop unneccessary cols/rows
print(trip_data.shape, stations.shape, sei.shape)
trip_data = trip_data.drop(columns=["rideable_type", "started_at", "ended_at","start_station_name", "start_station_id", "end_station_name", "end_station_id", "member_casual"])
stations = stations.drop(columns=['Short Name'])
sei = sei.drop(columns=['Layer'])
sei = sei.drop(index=0)
print(trip_data.shape, stations.shape, sei.shape)

# Rename cols
sei = sei.rename(columns={"SGX_2020-2024": "SEI"})

# Convert a string to number
sei["SEI"] = pd.to_numeric(sei["SEI"], errors="coerce")

# Q2 (ANAH)
# dropping unneeded columns
bike_routes = bike_routes.drop(columns=['MI_CTRLINE', 'CONTRAFLOW'])
bike_crashes = bike_crashes.drop(columns=['CRASH_DATE_EST_I', "NOT_RIGHT_OF_WAY_I",'HIT_AND_RUN_I',"DAMAGE", 'INJURIES_FATAL', 'INJURIES_INCAPACITATING', "INJURIES_NON_INCAPACITATING",'INTERSECTION_RELATED_I','INJURIES_REPORTED_NOT_EVIDENT', "INJURIES_NO_INDICATION", "INJURIES_UNKNOWN", 'CRASH_DATE', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'LANE_CNT', "ALIGNMENT","ROADWAY_SURFACE_COND","ROAD_DEFECT","REPORT_TYPE", "DATE_POLICE_NOTIFIED", 
                                          "BEAT_OF_OCCURRENCE","PHOTOS_TAKEN_I","STATEMENTS_TAKEN_I","DOORING_I","WORK_ZONE_I","WORK_ZONE_TYPE","WORKERS_PRESENT_I","NUM_UNITS","CRASH_MONTH", "CRASH_HOUR","CRASH_DAY_OF_WEEK","IDOT_CONTROL_NO"])

# taking out crashes that don't occur in the street where the bike lines are
bike_crashes[(bike_crashes['TRAFFICWAY_TYPE'] != 'ALLEY') & (bike_crashes['TRAFFICWAY_TYPE'] != 'PARKING LOT')]

# handling missing values

print(bike_routes.isna().sum())

print(bike_crashes.isna().sum())
# there is no way to fill in this information with the data provided, and it was only 33 out of 18,712 rows.
bike_crashes = bike_crashes.dropna(subset='LOCATION')

print(communities.isna().sum())

print(census.isna().sum())

# all significant missing values have been dealt with

# renaming columns to prepare for merge
bike_routes = bike_routes.rename(columns={'ST_NAME':'STREET_NAME'})

# Q3 (Wiktoria)

# Since the cordinates are stored as text, we need to convert them into a polygon object 
# Storing the polygon text into a variable named polygon
polygon = epa_df.columns[1] 

# Converting each text into a polygon object (with shape so spacial operations can used) in a new column named geometry
epa_df["geometry"] = epa_df[polygon].apply(wkt.loads)

# Making the table into a dataframe with geometry and a coordinate reference system (CRS) 
epa = gpd.GeoDataFrame(epa_df, geometry="geometry", crs="EPSG:4326")

# Making a copy of the divvy dataset to work with
clean_divvy = divvy_trips_df.copy()

# Shape before data cleaning
print("Before Cleaning:", clean_divvy.shape[0])

# Since all trips are either electric_bike or classic_bike don't need to drop any values
# Converting the timestamps to datetimes, no N/A values so don't have to drop any values
clean_divvy["started_at"] = pd.to_datetime(clean_divvy["started_at"], errors="coerce")
clean_divvy["ended_at"] = pd.to_datetime(clean_divvy["ended_at"], errors="coerce")

# To account for any "unrealistic" data such as trips that are under 1 min or over a day, we will drop any trips that are under 1 min or over 24 hours
clean_divvy["duration_min"] = (clean_divvy["ended_at"] - clean_divvy["started_at"]).dt.total_seconds() / 60
clean_divvy = clean_divvy[(clean_divvy["duration_min"] >= 1) & (clean_divvy["duration_min"] <= 24 * 60)]

# Shape after data cleaning
print("After Cleaning:", clean_divvy.shape[0])

# Q4 (Luke)

# Convert dates to datetime
rides['started_at'] = pd.to_datetime(rides['started_at'])
rides['ended_at'] = pd.to_datetime(rides['ended_at'])
Bears['Date'] = pd.to_datetime(Bears['Date'], format='%d/%m/%Y %H:%M', utc=True)    
Bears['Date'] = Bears['Date'].dt.tz_convert('America/Chicago')
Bulls['Date'] = pd.to_datetime(Bulls['Date'], format = '%d/%m/%Y %H:%M', utc = True)
Bulls['Date'] = Bulls['Date'].dt.tz_convert('America/Chicago')
Cubs['START DATE'] = pd.to_datetime(
    Cubs['START DATE'] + ' ' + Cubs['START TIME'],
    format='%m/%d/%y %I:%M %p'
)
Cubs['START DATE'] = Cubs['START DATE'].dt.tz_localize('America/Chicago')

# filter for games within the window (going back one calendar year)
start = pd.Timestamp("2025-11-01", tz="America/Chicago")
end = pd.Timestamp("2026-11-01", tz="America/Chicago")   

Bears = Bears[(Bears['Date'] >= start) & (Bears['Date'] < end)]
print(len(Bears))
Bulls = Bulls[(Bulls['Date'] >= start) & (Bulls['Date'] < end)]
print(len(Bulls))
Cubs = Cubs[(Cubs['START DATE'] >= start) & (Cubs['START DATE'] < end)]
print(len(Cubs))

# check where duplicates are, and drop exact duplicates
print(rides.duplicated(subset = ['ride_id']).sum())
print(rides.duplicated().sum())
rides = rides.drop_duplicates()
print(rides.duplicated(subset = ['ride_id']).sum())

# Checks to make sure everything is within the window, and there are no impossible lat/long values
print(rides['started_at'].min())
print(rides['started_at'].max())
rides['month'] = rides['started_at'].dt.month
print(rides['start_lat'].max())
print(rides['start_lat'].min())
print(rides['start_lng'].max())
print(rides['start_lng'].min())
print(rides['end_lat'].max())
print(rides['end_lat'].min())
print(rides['end_lng'].max())
print(rides['end_lng'].min())
# Remove any impossible trips which end before they start

valid = (rides['ended_at'] > rides['started_at']) 
rides = rides[valid].copy()

# Create a new variable representing the duration of a trip
rides['duration'] = rides['ended_at'] - rides['started_at']
rides['duration_min'] = rides['duration'].dt.total_seconds() / 60

# Filter outlier trips that were less than 1 minute, and longer than 240. Combined, these trips only account for around 2.7 % of trips
print(rides['duration_min'].describe())
too_short = rides['duration_min'] < 1
too_long = rides['duration_min'] > 240 
print(len(rides[too_long]) / len(rides) * 100)
print(len(rides[too_short]) / len(rides) * 100)
keep = ~(too_short | too_long)
rides = rides[keep].copy()
print(rides['duration_min'].describe()) 

# Remove trips missing a station id. To measure departures by station, dockless trips must be removed. 
known_station_id = rides.dropna(subset = 'start_station_id').copy()
known_station_id['date'] = known_station_id['started_at'].dt.normalize()

# Remove the unnecessary cols from each Team's dataset
Cubs = Cubs.rename(columns = {'LOCATION': 'Location', 'START DATE': 'Date'})
Cubs['Home Team'] = 'Chicago Cubs'
Cubs = Cubs.drop(columns = ['START TIME', 'START TIME ET', 'SUBJECT', 'DESCRIPTION', 'END DATE', 'END DATE ET', 'END TIME', 'END TIME ET', 
                            'REMINDER OFF', 'REMINDER ON', 'REMINDER DATE', 'REMINDER TIME', 'REMINDER TIME ET', 'SHOWTIMEAS FREE', 'SHOWTIMEAS BUSY'])
Bears = Bears.drop(columns = ['Match Number', 'Round Number', 'Away Team', 'Result'])
Bulls = Bulls.drop(columns = ['Match Number', 'Round Number', 'Away Team', 'Result'])
# Combine the games into one data set, and normalize the date
Sports = pd.concat([Bears, Bulls, Cubs], ignore_index=True)
Sports['date'] = Sports['Date'].dt.tz_localize(None).dt.normalize()

**Cleaning summary.** In a few bullets:

- What you have cleaned so far (missing values, duplicates, data types, outliers, merges, new variables):

For Q1: cleaned by dropping unnecessary columns and rows, converted a column datatype, renamed a column to be easier to handle.

For Q2: I cleaned up bike route and crash information by dropping unecessary columns and dropping null values that could not be filled and renamed a column to handle merge. I saw no missing values in the community and census data except for one value in the community areas, but I found that insignificant so I kept it.

For Q3: Changed the text of the epa dataset that holds the coordinates of the EPA area into into a polygon object in a new column named geometry so I can do spatial operations on it later. Converted all the timestamps to datetimes and filtered out divvy trips that are under 1 min or over a day long.

For Q4: Dropped unnecessary cols, converted datetimes, filtered to my window, dropped duplicates, removed impossible trips, trips <1 minute and > 240 mins, removed trips without station id. 

- What cleaning is still left:

Q1: There are 644 missing values in end coordinates in trip_data, but cannot be imputed, so that data should be handled carefully.

Q2: I need to drop uneccesary columns from the community and census data.

Q3: For station-level analysis, drop any trips with no station_id for start time.

Q4: might need to drop stations with low traffic baseline, create a new flag 'weekend' to compute weekend baseline. 

- Rows and columns before cleaning → after cleaning: 

Q1: trip data: (770562, 13), stations: (1231, 9), SEI: (78, 7) --> (770562, 5), (1231, 8), (77, 6)

Q2: 

Q3: divvy data: (770562, 13), (751319, 13), epa data: (1, 4), (1,5)

Q4:divvy data: (6294451, 13) -> (4907722, 16) Bears (19, 7) , Bulls (82, 7) , Cubs(179, 17) -> concatenated to Sports (129, 4) 

## 2) Early results
Show **at least one** early result or plot **for each team member's question**. It does not need to be final or polished, but it should come from your real data.

Under each result, write one or two sentences on what it shows so far.

### Question 1: Is there a correlation between Social Engagement Index and where Divvy trips and stations are concentrated?
*By Yujin Tatar*

In [ ]:
# Early analysis or plot for question 1
# Seeing which neighborhoods have high and low SEI:

# Selecting the top 10 and bottom 10 SEI and combining them
top10 = sei.nlargest(10, "SEI")
bottom10 = sei.nsmallest(10, "SEI")
both = pd.concat([bottom10, top10]).sort_values("SEI")

# Making a horizontal bar plot since the neighborhood names are long, to represent the SEI visually
plt.figure(figsize=(8, 7))
plt.barh(both["Name"], both["SEI"])
plt.xlabel("Social Engagement Index (2020-2024)")
plt.title("Chicago Neighborhoods with the Lowest and Highest SEI")
plt.tight_layout()
plt.show()

**What this shows so far:**

The distribution of neighborhoods with high and low SEI. I will see if these neighborhoods are also the ones that have high/low concentration of trips and stations.

### Question 2: Does the most prominent type of bike lane affect bike usage? Which types of lanes have the most crashes? Compare against the hardship index ?
*By Anah Shaikh*

In [ ]:
# Early analysis or plot for question 2
# Seeing how many crashes each bike lane type has over entire data timeline

# Merging data by street name
bike_merged = bike_crashes.merge(bike_routes, on='STREET_NAME')

# Grouping Data
crash_counts = bike_merged.groupby('DISPLAYROU').size().reset_index(name='num_crashes')

# Visualizing Data
plt.figure(figsize=(12, 6))
sns.barplot(data=crash_counts, x='DISPLAYROU', y='num_crashes')
plt.xlabel('Type of Bike Lane')
plt.ylabel('Number of Crashes')
plt.title('Crashes per Type of Bike Lane')
plt.show()

**What this shows so far:** 
This shows that the Buffered Bike Lane has the most amount of crashes, with the Neighborhood Greenway trailing a decent way behind. The next step would be to determine if there's a pattern or trend, perhaps looking at if the amount of crashes increases with the amount of usage on certain bike routes.

### Question 3: Does the share of trips by bike type differ between Divvy stations inside/outside Equity Priority Areas?
*By Wiktoria Mleczko*

In [ ]:
# Early analysis or plot for question 3

# Marking each trip as EPA or non-EPA via a spatial join

# Turning each point into a point on the map using the start coordinates of each trip 
trip_points = gpd.GeoDataFrame(
    clean_divvy.copy(),
    geometry=gpd.points_from_xy(clean_divvy["start_lng"], clean_divvy["start_lat"]),
    crs="EPSG:4326",
)

# Checking each trip point against an EPA boundary to see if it is inside
trip_epa = gpd.sjoin(trip_points, epa[["geometry"]], how="left", predicate="within")

# Turning the match column into a boolean column indicating if each trip is inside an EPA boundary
clean_divvy["in_epa"] = trip_epa["index_right"].notna().values

print(clean_divvy["in_epa"].value_counts())


# ------------------------------------------------------
area_names = {False: "Outside EPA", True: "Inside EPA"}

# Flag trips that have a start station ID
clean_divvy["has_station"] = clean_divvy["start_station_id"].notna()

# One row per station with its location and EPA label, for the map
station_coords = (
    clean_divvy.dropna(subset=["start_station_id"])
    .groupby("start_station_id")[["start_lat", "start_lng", "in_epa"]]
    .first()
    .reset_index()
)
stations_df = gpd.GeoDataFrame(
    station_coords,
    geometry=gpd.points_from_xy(station_coords["start_lng"], station_coords["start_lat"]),
    crs="EPSG:4326",
)

# Chicago background outline
chicago = gpd.read_file("data/chicago_commpop.geojson")

#------------------------------------------------------------
# Graph 1: Making a map of the Divvy stations relative to the EPA area

chicago = gpd.read_file("data/chicago_commpop.geojson")

fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")

# Creating the background: city outline
chicago.plot(ax=ax, color="lightgrey", edgecolor="dimgray", linewidth=0.5)
# Shading the EPA area
epa.plot(ax=ax, color="lightblue", edgecolor="navy", linewidth=1.5, alpha=0.6)

# Shading stations so grey for outside the EPA, blue for inside the EPA
stations_df[~stations_df["in_epa"]].plot(
    ax=ax, color="dimgray", markersize=6, alpha=0.5, label="Outside EPA"
)
stations_df[stations_df["in_epa"]].plot(
    ax=ax, color="#1466a0", markersize=6, alpha=0.9, label="Inside EPA"
)

# Setting the title, axis labels, aspect ratio, legend, and grid
ax.set_title("Divvy stations relative to Equity Priority Area",
             fontsize=15, fontweight="bold", color="#222222", loc="left", pad=30)
ax.text(0, 1.02, "Trip start stations, Chicago, Sept. 2026 sample period",
        transform=ax.transAxes, fontsize=10, color="#555555", ha="left")
ax.set_aspect("equal")
ax.set(xlabel="Longitude", ylabel="Latitude")
ax.legend(loc="lower left", frameon=True, title="Station location")
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.show()

#---------------------------------------------------------------------------------
# Graph #2: Showing the share of trips starting inside vs. outside the EPA

# counting the # of trips + index
counts = clean_divvy["in_epa"].value_counts().sort_index()
# getting the labels for each category
labels = [area_names[k] for k in counts.index]
# getting total number of trips
total = counts.sum()

fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")

# Making the pie chart
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=labels,
    colors=["#cfe3ef", "#1f4e79"],
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.1f}%",
    pctdistance=1.25,                      
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 11},
)

# Text for labels
for t in autotexts:
    t.set_color("#222222")

ax.text(0.75, 1.0, f"{total:,} trips in total", transform=ax.transAxes,
        fontsize=10, color="#555555", ha="left")

plt.show()

**What this shows so far:**
So far this shows that there are more stations outside the EPA than inside, and most tations clustered in north/downtown. Additionally, most trips (about 98%) start outside the EPA. Next I will look at the bike type of these trips.


### Question 4: \<How do different gamedays for Chicago sports teams affect bike demand?\>
*By \<Luke Hallum\>*

In [ ]:

# Early analysis or plot for question 4
# Create a table of trips by station. Median lat/long in case some coords vary slightly. 
# First and last dates included to tell when a station was active. 
stations = known_station_id.groupby('start_station_id').agg(
    Station_name= ('start_station_name', 'first'),
    median_lat = ('start_lat' , 'median'), 
    median_lng = ('start_lng', 'median'),
    total_rides = ('ride_id', 'count'), 
    first_date = ('started_at', 'min'),
    last_date = ('started_at', 'max')

)
stations = stations.reset_index()

# Convert dates so they will match with the dates in the other dfs. 
stations['first_date'] = stations['first_date'].dt.normalize()
stations['last_date'] = stations['last_date'].dt.normalize()

# Chack that stations matches the number of unique known stations. 
print(len(stations) == known_station_id['start_station_id'].nunique())

# create a new variable that measures the haversine distance between each station and each sports venue

stations['dist_SF'] = stations.apply(
    lambda row: haversine(list((row['median_lat'], row['median_lng'])) , Soldier, unit = Unit.MILES), 
    axis = 1
)
stations['dist_Wrig'] = stations.apply(
    lambda row: haversine(list((row['median_lat'], row['median_lng'])) , Wrigley, unit = Unit.MILES), 
        axis = 1
)
stations['dist_UC'] = stations.apply(
    lambda row: haversine(list((row['median_lat'], row['median_lng'])) , United, unit = Unit.MILES), 
        axis = 1
)
df = known_station_id.groupby(['start_station_id', 'date']).agg(
    departures = ('ride_id', 'count'), 
)
df = df.reset_index()
# Create flags for stations within .5 miles of each venue
radius = .5

stations['near_SF']   = stations['dist_SF']   <= radius
stations['near_Wrig'] = stations['dist_Wrig'] <= radius
stations['near_UC']   = stations['dist_UC']   <= radius

print(stations[['near_SF', 'near_Wrig', 'near_UC']].sum())
# create a df of each date within my window to match with stations near each venue
window = pd.date_range(start="2025-11-01", end="2026-10-31", freq="D")
days = pd.DataFrame({'date':window})
nearby = stations[stations[['near_SF', 'near_Wrig', 'near_UC']].any(axis = 1)].copy()
grid = nearby.merge(days, how = 'cross')

# Keep only dates within when each station was active, then fill days with no trips as 0 departures. 
active = ((grid['date'] >= grid['first_date']) & (grid['date'] <= grid['last_date']))
grid = grid[active].copy()
new_grid = grid.merge(df[['start_station_id', 'date', 'departures']], on = ['start_station_id', 'date'], how = 'left')
new_grid['departures'] = (new_grid['departures'].fillna(0)).astype(int)
new_grid['month'] = new_grid['date'].dt.month
cubs_dates = Sports.loc[Sports['Home Team'] == 'Chicago Cubs', 'date']
event_Wrig = new_grid['near_Wrig'] & new_grid['date'].isin(cubs_dates)


bears_dates = Sports.loc[Sports['Home Team'] == 'Chicago Bears', 'date']
event_SF = new_grid['near_SF'] & new_grid['date'].isin(bears_dates)


bulls_dates = Sports.loc[Sports['Home Team'] == 'Chicago Bulls', 'date']
event_UC = new_grid['near_UC'] & new_grid['date'].isin(bulls_dates)

# create a new col representing whether or not that day was a gameday for any venue
new_grid['is_event'] = event_SF | event_UC | event_Wrig

# Get normal days and define a baseline number of departures by station by month
normal_days = new_grid[~new_grid['is_event']] 

baseline = normal_days.groupby(['start_station_id', 'month']).agg(
    base_departures = ('departures', 'mean'),
    n_days = ('date', 'count')
)
baseline = baseline.reset_index()

event_days = new_grid[new_grid['is_event']]

# merge the baseline df and the event_day line for analysis

base_vs_event = event_days.merge(baseline, on = ['start_station_id','month'], how = 'left')
# Check how manyinstances where a station had 0 departures 
zero = base_vs_event['base_departures'] == 0 
print(zero.sum())
# Remove them to avoid a divide by zero error
base_vs_event = base_vs_event.loc[base_vs_event['base_departures']> 0].copy()

# Calculate the percent change in departures on a gameday compared to a baseline day
base_vs_event['pct_lift'] = ((base_vs_event['departures'] - base_vs_event['base_departures']) / base_vs_event['base_departures']) * 100

# label each event row with a team based on which venue flag it is near
conditions = [base_vs_event['near_SF'], base_vs_event['near_UC'], base_vs_event['near_Wrig']]
name = ['Bears', 'Bulls', 'Cubs']
base_vs_event['team'] = np.select(conditions, name, default = 'Other')

# make sure no team got an 'Other' label
base_vs_event['team'].value_counts()

# Measure lift by team
lift = base_vs_event.groupby('team').agg(
    median_lift = ('pct_lift', 'median'),
    mean_lift = ('pct_lift', 'mean'),
    lift_counts = ('pct_lift', 'count')
)

lift.head(3)

# plot median pct change in lift by team
sns.barplot(data = lift, x = 'team', y = 'median_lift')
plt.ylabel('Median % change in departures vs normal days')
plt.xlabel('Team')
plt.axhline(color = 'red')
plt.show()



**What this shows so far:**

On a Cubs gameday, nearby stations see about 60% more departures compared to a non gameday
On a Bears gameday, nearby stations see about 40% more departures compared to a non gameday
On a Bulls gameday, nearby stations see about 20% less departures compared to a non gameday. 

## 3) What remains
List what is left to finish before your presentation and report. Give each item an owner and a target date.

| Task | Owner | Target date |
|---|---|---|
|Finish analyzing where the trips and stations are and then see the correlation with SEI|Yujin |11/24 |
|Finish analyzing possible correlations between bike lane types and crashes, then see how this maps onto socioeconomic indicators to see if bike lane types are affecting these communities in different ways. |Anah|11/24 |
|Finish creating other visulizations and determine how to compare/adjust station level values to trip level data |Wiktoria |11/24|
|Create the following: Lift by month per team, Top 10 stations by median lift, Distance decay plot for lift per team |Luke |11/24|

**Biggest risk to finishing on time** (one sentence): The biggest risk is putting all our data together and ensuring it all works for our presentation on Wednesday. 

## 4) Questions for the mentor
List the specific questions or problems you want help with. "Is this okay?" is too general. Say what you tried and what you are unsure about.

For example: *"Our income column is very right-skewed. Should we use a log scale in the plot or report medians instead of means?"*

1. (Yujin) For the question about social engagement index, should I also additionally look at other SES indices as well included in the census csv file that was provided as an external dataset? Like maybe I could do a correlation heatmap with those other indices as well?
2. (Anah) I merged on street names because I had the issue of having to do a bunch of things that we didn't learn trying to handle the geom. I wanted to initially see more clearly where the crashes were so I could tell exactly what route it was on, but after AI assistance with figuring out the packages, I had a very small DataFrame left after merging. Does merging on the street name make my data less accurate?
3. (Wiktoria) My question asks if the difference still shows up when I adjust for the number of stations. I dropped the trips without a station ID because I figured those were rides picked up or dropped off away from a station, but I'm not sure if dropping them counts as adjusting for station count, or if I need a different method. Could you tell me what adjustment you're looking for?
4. (Luke) I excluded around 20% of trips because they had no station ID, is that a reasonable choice for a question about demand around venues? My departures baseline is non gamedays by month, should I add a flag for the weekend? Some stations have very small baselines, which makes some stations have extreme lifts. In my plots I reported medians, is that reasonable, or should I drop low-traffic stations instead? 

## Mentor feedback *(fill in during or right after the check-in)*

Write down what the mentor suggested so the whole team has a record.

-
-